# Interactive Regression Models: Binary Treatments with DML

**Goal:** Estimate ATE and ATTE for a binary treatment using `DoubleMLIRM`, compare to PLR, and run propensity score diagnostics.

**Time:** 15 minutes

You will estimate the effect of sanctions on shipping freight rates,
compare ATE vs ATTE, and check propensity score overlap.

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

In [ ]:
# Apply course styling
apply_course_theme()
apply_plot_theme()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from doubleml import DoubleMLData, DoubleMLIRM, DoubleMLPLR
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')

## Simulate: Sanctions on Shipping Freight Rates

Binary treatment (sanctioned route vs not) with heterogeneous effects.

In [ ]:
n = 3000
p = 30

X = np.random.randn(n, p)
col_names = [f'X{i}' for i in range(p)]

# Propensity score
propensity = 1 / (1 + np.exp(-(0.5 * X[:, 0] + 0.3 * X[:, 1] - 0.2 * X[:, 3])))
D = np.random.binomial(1, propensity)

# Potential outcomes with heterogeneous treatment effect
Y_0 = 2.0 + 0.5 * X[:, 0] + 0.3 * X[:, 2] + np.random.randn(n) * 0.5
Y_1 = Y_0 + 1.5 + 0.8 * X[:, 0]  # Effect varies with X[0]
Y = D * Y_1 + (1 - D) * Y_0

true_ate = np.mean(Y_1 - Y_0)
true_atte = np.mean((Y_1 - Y_0)[D == 1])

print(f'True ATE:  {true_ate:.3f}')
print(f'True ATTE: {true_atte:.3f}')
print(f'Treated:   {D.sum()} ({D.mean():.1%})')
print(f'\nATE != ATTE because treated routes have higher X[0] (more affected).')

## Estimate ATE and ATTE with IRM

Use `DoubleMLIRM` with random forest nuisance models.

In [ ]:
df = pd.DataFrame(X, columns=col_names)
df['freight_premium'] = Y
df['sanctions'] = D

dml_data = DoubleMLData(df, y_col='freight_premium',
                        d_cols='sanctions', x_cols=col_names)

# ATE
irm_ate = DoubleMLIRM(dml_data,
    ml_g=RandomForestRegressor(200, random_state=42),
    ml_m=RandomForestClassifier(200, random_state=42),
    score='ATE', n_folds=5, trimming_threshold=0.05)
irm_ate.fit()

# ATTE
irm_atte = DoubleMLIRM(dml_data,
    ml_g=RandomForestRegressor(200, random_state=42),
    ml_m=RandomForestClassifier(200, random_state=42),
    score='ATTE', n_folds=5, trimming_threshold=0.05)
irm_atte.fit()

print('ATE Results:')
print(irm_ate.summary)
print(f'\nATTE Results:')
print(irm_atte.summary)
print(f'\nTrue ATE:  {true_ate:.3f}')
print(f'True ATTE: {true_atte:.3f}')

## Compare IRM to PLR

PLR assumes a constant effect. With heterogeneity, PLR gives a different estimate.

In [ ]:
plr = DoubleMLPLR(dml_data,
    ml_l=RandomForestRegressor(200, random_state=42),
    ml_m=RandomForestRegressor(200, random_state=42),
    n_folds=5)
plr.fit()

print('Comparison:')
print(f'  PLR (constant):  {plr.coef[0]:.3f}')
print(f'  IRM ATE:         {irm_ate.coef[0]:.3f}')
print(f'  IRM ATTE:        {irm_atte.coef[0]:.3f}')
print(f'  True ATE:        {true_ate:.3f}')
print(f'  True ATTE:       {true_atte:.3f}')
print(f'\nIRM separates ATE from ATTE; PLR gives a weighted average.')

## Visualise: ATE vs ATTE Forest Plot

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

estimates = {
    'PLR (constant)': (plr.coef[0], plr.confint().iloc[0, 0], plr.confint().iloc[0, 1]),
    'IRM ATE': (irm_ate.coef[0], irm_ate.confint().iloc[0, 0], irm_ate.confint().iloc[0, 1]),
    'IRM ATTE': (irm_atte.coef[0], irm_atte.confint().iloc[0, 0], irm_atte.confint().iloc[0, 1]),
}

for i, (name, (coef, lo, hi)) in enumerate(estimates.items()):
    ax.errorbar(coef, i, xerr=[[coef - lo], [hi - coef]],
                fmt='o', capsize=6, markersize=10, linewidth=2)

ax.axvline(x=true_ate, color='blue', linestyle='--', alpha=0.7, label=f'True ATE={true_ate:.2f}')
ax.axvline(x=true_atte, color='red', linestyle='--', alpha=0.7, label=f'True ATTE={true_atte:.2f}')
ax.set_yticks(range(len(estimates)))
ax.set_yticklabels(list(estimates.keys()), fontsize=11)
ax.set_xlabel('Treatment Effect', fontsize=12)
ax.set_title('PLR vs IRM: Binary Treatment Estimates', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

## Summary

**What you learned:**
1. IRM extends DML to binary treatments with heterogeneous effects
2. ATE averages over all units; ATTE averages over treated units only
3. PLR gives a weighted average that may not match either ATE or ATTE
4. Propensity score quality and overlap are essential for IRM reliability

**What is next:**
- **Module 07:** Instrumental variables with DML (PLIV)
- **Module 08:** Heterogeneous treatment effects (CATE) with `econml`

**Key takeaway:** For binary treatments, use IRM. It separates ATE from ATTE
and handles heterogeneous effects that PLR misses.

In [ ]:
learning_objectives(["IRM extends DML to binary treatments with heterogeneous effects", "ATE averages over all units; ATTE averages over treated units only", "PLR gives a weighted average that may not match either ATE or ATTE", "Propensity score quality and overlap are essential for IRM reliability"])

In [ ]:
key_takeaways([
    "Core concept from this notebook demonstrated with working code",
    "Key parameters and their effects on results",
    "When to apply this technique vs alternatives"
])